# Build a Baseline Classification Helper

In this lesson, you build a simple helper that uses an LLM to pick a **baseline classifier**. The LLM is an assistant, not the trainer. It looks at a short profile of your data and suggests **classifiers worth trying**.

![](../../images/classification_helper_flow.png)

## 1 - Setup

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -q google-genai pandas scikit-learn python-dotenv

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../02-Preprocessing-Helpers").resolve()))


In [ ]:
import importlib
import json
import os
import re
import warnings

import pandas as pd
from dotenv import load_dotenv
from google import genai
from preprocessing_pipeline import preprocessing_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

warnings.filterwarnings("ignore")

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

GEN_CONFIG = {"temperature": 0.0, "seed": 42}

In [ ]:
result = preprocessing_pipeline(
    raw_path="../../data/hr_analytics.csv",
    target_col="Attrition",
    task="classification",
)
X_train = result.X_train_enc
X_test = result.X_test_enc
y_train = result.y_train
y_test = result.y_test

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Target mean (train): {y_train.mean():.2f}")


## 2 - Profile the Data

Before asking the LLM anything, we build a short profile of the training data: size, class balance, how correlated the features are, and a quick logistic regression score as a rough difficulty signal. This gives the LLM enough context to make a sensible suggestion.


In [ ]:
def profile_data(X_train, y_train):
    """Build a short profile the LLM can reason about."""
    X_df = pd.DataFrame(X_train)
    pos_ratio = float(pd.Series(y_train).mean())

    # dtype mix — many categoricals favor trees, all-numeric favors linear models
    n_numeric = int(X_df.select_dtypes(include="number").shape[1])
    n_categorical = int(X_df.shape[1] - n_numeric)

    # max absolute skew across numeric features — heavy skew favors trees
    numeric = X_df.select_dtypes(include="number")
    skew_max = float(numeric.skew().abs().max()) if numeric.shape[1] else 0.0

    # near-constant features (hint of sparse/one-hot columns)
    low_var = int((numeric.var() < 0.01).sum()) if numeric.shape[1] else 0

    # rows-to-features ratio — low values favor regularized linear models
    n_to_p_ratio = round(X_df.shape[0] / max(X_df.shape[1], 1), 2)

    # how correlated are the features (a hint of multicollinearity)
    corr = X_df.corr().abs().to_numpy(copy=True)
    np.fill_diagonal(corr, np.nan)
    max_corr = float(np.nanmax(corr))

    # two quick probes — a linear one and a shallow tree one.
    # The gap between them hints at how non-linear the problem is.
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    linear_probe = LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    )
    tree_probe = RandomForestClassifier(
        n_estimators=100,
        max_depth=6,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )
    linear_auc = float(
        cross_val_score(linear_probe, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    )
    tree_auc = float(
        cross_val_score(tree_probe, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    )

    return {
        "rows": int(X_df.shape[0]),
        "features": int(X_df.shape[1]),
        "n_numeric": n_numeric,
        "n_categorical": n_categorical,
        "n_to_p_ratio": n_to_p_ratio,
        "positive_class_ratio": round(pos_ratio, 3),
        "max_feature_correlation": round(max_corr, 3),
        "skewness_max": round(skew_max, 3),
        "low_variance_features": low_var,
        "logreg_probe_roc_auc": round(linear_auc, 3),
        "tree_probe_roc_auc": round(tree_auc, 3),
    }


In [ ]:
profile = profile_data(X_train, y_train)
profile


## 3 - Ask the LLM for 3 Candidates

We give the LLM the profile and ask it to suggest **three classifiers** that fit this dataset. For each one, it returns:
- `module` + `class` — the full import path (e.g. `sklearn.ensemble.RandomForestClassifier`)
- `reason` — grounded in specific profile values
- `search_space` — hyperparameter ranges Optuna will explore

No fixed allowlist. The LLM can pick anything sklearn-compatible.

In [ ]:
SELECTION_PROMPT = (
    "You are a machine learning engineer. "
    "Based ONLY on the dataset profile below (not general ML advice), suggest EXACTLY 3 "
    "classifiers to try as baselines. You may choose ANY scikit-learn-compatible classifier "
    "(sklearn.*, xgboost.XGBClassifier, etc). Pick the three you think best fit THIS profile.\n\n"
    "For each candidate return:\n"
    "  - module: full import path, e.g. 'sklearn.ensemble'\n"
    "  - class:  class name,        e.g. 'RandomForestClassifier'\n"
    "  - reason: MUST cite specific field names and values from the profile. "
    "If the reason could apply to any dataset, rewrite it.\n"
    "  - init_kwargs: a dict of constructor kwargs to use as-is (e.g. random_state=42, "
    "class_weight='balanced' when positive_class_ratio is low). Keep this minimal — we are "
    "training baselines with defaults, not tuning.\n\n"
    'Return ONLY valid JSON: {"candidates": [{"module": str, "class": str, "reason": str, '
    '"init_kwargs": {...}}, ...]}'
)

In [ ]:
def pick_candidates(profile):
    """Ask the LLM for 3 classifiers. Validate by actually importing the class."""
    content = f"Profile: {json.dumps(profile)}"
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=content,
        config={"system_instruction": SELECTION_PROMPT, **GEN_CONFIG},
    )
    text = re.sub(r"```(?:json)?", "", response.text or "").replace("```", "").strip()
    picks = json.loads(text)["candidates"]
    assert len(picks) == 3, f"Expected 3 candidates, got {len(picks)}"
    for p in picks:
        # Validate the class is real by importing it — guardrail against hallucination.
        mod = importlib.import_module(p["module"])
        p["cls"] = getattr(mod, p["class"])
        p.setdefault("init_kwargs", {})
        # Pin random_state so scores are reproducible regardless of what the LLM returned.
        p["init_kwargs"]["random_state"] = 42
    return picks

In [ ]:
candidates = pick_candidates(profile)
for c in candidates:
    print(f"- {c['module']}.{c['class']}  init_kwargs={c['init_kwargs']}")
    print(f"    reason: {c['reason']}")

## 4 - Score Each Candidate with Cross-Validation

For each of the 3 classifiers, we fit with the LLM's suggested `init_kwargs` and score with 3-fold CV on ROC AUC. No hyperparameter tuning here — that comes in lesson 3.3. We just want a quick baseline leaderboard.

In [ ]:
def score_candidate(candidate, X_train, y_train):
    """Fit + 3-fold CV on ROC AUC. No tuning."""
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    clf = candidate["cls"](**candidate["init_kwargs"])
    roc_auc = cross_val_score(
        clf, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1
    ).mean()
    return {
        "model": f"{candidate['module']}.{candidate['class']}",
        "cls": candidate["cls"],
        "init_kwargs": candidate["init_kwargs"],
        "roc_auc": roc_auc,
    }

In [ ]:
leaderboard = (
    pd.DataFrame([score_candidate(c, X_train, y_train) for c in candidates])
    .sort_values("roc_auc", ascending=False)
    .reset_index(drop=True)
)

leaderboard[["model", "roc_auc", "init_kwargs"]]

## 5 - Fit the Winner

The top row of the leaderboard is our baseline. We refit it on the full training set. We report one headline number (ROC AUC) here — deep evaluation (metrics interpretation, thresholds, anomalies) lives in Module 4.


In [ ]:
winner = leaderboard.iloc[0]
model = winner["cls"](**winner["init_kwargs"])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

In [ ]:
print(f"{winner['model']} — test ROC AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(
    "Full evaluation (precision/recall/threshold tuning/anomalies) happens in Module 4."
)


## 6 - Wrap It All in One Function

We just walked through the steps one by one: profile the data, ask the LLM for 3 candidates, tune each with Optuna, pick the winner, fit, and score. Now let's wrap that whole flow into a single function `classification_helper` so we can reuse it in later lessons with one call.

In [ ]:
def classification_helper(X_train, X_test, y_train, y_test):
    """Run the full flow: profile → LLM picks 3 → CV-score → fit winner → return results."""
    profile = profile_data(X_train, y_train)
    candidates = pick_candidates(profile)

    leaderboard = (
        pd.DataFrame([score_candidate(c, X_train, y_train) for c in candidates])
        .sort_values("roc_auc", ascending=False)
        .reset_index(drop=True)
    )

    winner = leaderboard.iloc[0]
    model = winner["cls"](**winner["init_kwargs"])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = (
        model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    )

    return {
        "model": model,
        "model_name": winner["model"],
        "init_kwargs": winner["init_kwargs"],
        "profile": profile,
        "candidates": candidates,
        "leaderboard": leaderboard,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

In [ ]:
result = classification_helper(X_train, X_test, y_train, y_test)
print("Winner:", result["model_name"])
print("Test ROC AUC:", round(roc_auc_score(y_test, result["y_proba"]), 3))

## 7 - Save for Reuse

Dump the helper (plus its imports and the functions it depends on) into `classification_helper.py` so later lessons can `from classification_helper import classification_helper`.

In [ ]:
import inspect

components = [
    "import importlib",
    "import json",
    "import os",
    "import re",
    "import numpy as np",
    "import pandas as pd",
    "from dotenv import load_dotenv",
    "from google import genai",
    "from sklearn.ensemble import RandomForestClassifier",
    "from sklearn.linear_model import LogisticRegression",
    "from sklearn.model_selection import StratifiedKFold, cross_val_score",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"GEN_CONFIG = {GEN_CONFIG!r}",
    "",
    f"SELECTION_PROMPT = {SELECTION_PROMPT!r}",
    "",
    inspect.getsource(profile_data),
    "",
    inspect.getsource(pick_candidates),
    "",
    inspect.getsource(score_candidate),
    "",
    inspect.getsource(classification_helper),
]

with open("classification_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved classification_helper.py")